# 🧑‍🦲 光头强专属 RVC 音色模型 · 训练笔记本（Kaggle 免费 GPU 版）

> 训练自己的光头强音色转换模型（RVC v2），之后可用「微软情绪 TTS + RVC 转换」
> 做出带动漫配音感的光头强语音。**Kaggle 免费 GPU + 每次会话 12 小时**，政策比 Colab 宽松。

## ⚙️ 首次必做：开启免费 GPU

右上角 **Settings（设置）→ Accelerator（加速器）→ 选 GPU T4 x2 → 点击 Save（保存）**
（不设置会默认用 CPU，很慢。设置后下拉保存即可。）

## 📋 流程

1. 依次运行每个格子：装环境 → 下预训练模型 → 下素材 → 人声分离 → 切段 → 提特征 → **训练（约 30-60 分钟）** → 提取索引
2. 训练完在文件侧栏下载模型（gtq.pth + index）
3. 之后配合微软 TTS 无限次使用

## ⚠️ 重要提醒

- 免费用户每周约 30 小时 GPU，训练约 1 小时，够用多次
- 会话超时/闲置会自动断开，重开后重跑第 1、2 格即可
- 训练期间请勿关闭标签页


## 第 0 步：确认 GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(r)
print("GPU OK" if ("T4" in r or "Tesla" in r) else "请到 右上角Settings→Accelerator→GPU T4x2→Save 开启GPU后再重跑本格")

## 第 1 步：安装 RVC WebUI 环境（约 3 分钟）

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
%cd /kaggle/working/rvc
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip install -q demucs 2>&1 | tail -2
print("环境安装完成")

## 第 2 步：下载预训练模型（约 200MB）

In [ ]:
%cd /kaggle/working/rvc
import os
os.makedirs("assets/hubert_base", exist_ok=True)
os.makedirs("assets/rmvpe", exist_ok=True)
os.makedirs("assets/pretrained_v2", exist_ok=True)

!wget -q -O assets/hubert_base/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
!wget -q -O assets/rmvpe/rmvpe.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt
!wget -q -O assets/pretrained_v2/f0G40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth
!wget -q -O assets/pretrained_v2/f0D40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth

!ls -lh assets/hubert_base assets/rmvpe assets/pretrained_v2
print("预训练模型下载完成")

## 第 3 步：下载素材并解压

素材已托管在 GitHub，自动下载（**10 段精选光头强原声音频**，无需手动上传）。

In [ ]:
%cd /kaggle/working/rvc
!wget -q https://raw.githubusercontent.com/haiyulangman/gtq-voice/main/rvc_material_v2.zip -O rvc_material.zip
!unzip -o -q rvc_material.zip -d raw_material
!ls -lh raw_material/

## 第 4 步：人声分离（demucs，T4 上约 5-10 分钟）

In [ ]:
%cd /kaggle/working/rvc
import glob, os, shutil

files_in = glob.glob("raw_material/*.m4a") + glob.glob("raw_material/*.mp3") + glob.glob("raw_material/*.wav")
print("待分离:", [os.path.basename(f) for f in files_in])

os.makedirs("raw_vocals", exist_ok=True)
for f in files_in:
    !demucs --two-stems=vocals -n htdemucs "{f}" -o separated 2>&1 | tail -1

vocals = glob.glob("separated/htdemucs/**/vocals.wav", recursive=True)
print(f"分离出 {len(vocals)} 个人声文件")
for i, v in enumerate(vocals):
    shutil.copy(v, f"raw_vocals/vocals_{i:02d}.wav")
!ls -lh raw_vocals/

## 第 5 步：切段预处理（静音检测切 3.7 秒片段）

In [ ]:
%cd /kaggle/working/rvc
!python train/preprocess.py raw_vocals 40000 2 logs/gtq False 3.7 2>&1 | tail -5
import os
n = len(os.listdir("logs/gtq/0_gt_wavs"))
print(f"切段完成：共 {n} 个训练片段")

## 第 6 步：提取特征（F0 rmvpe + HuBERT，约 5-10 分钟）

In [ ]:
%cd /kaggle/working/rvc
!python train/dataset/extract_f0.py cuda 1 0 0 logs/gtq True 2>&1 | tail -3
!python train/dataset/extract_hubert_feature.py cuda 1 0 0 logs/gtq True v2 2>&1 | tail -3
print("特征提取完成")
!ls logs/gtq/

## 第 7 步：训练模型（约 30-60 分钟，请勿关闭页面）

- v2 架构、40k 采样率、batch 8、共 300 轮、每 50 轮保存一次
- 若提示显存不足，把 `-bs 8` 改成 `-bs 4` 重跑本格

In [ ]:
%cd /kaggle/working/rvc
!python train/train.py -e gtq -sr 40k -f0 1 -bs 8 -g 0 -te 300 -se 50 \
  -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth \
  -l 1 -c 0 -sw 1 -v v2 2>&1 | tail -20
print("训练结束")
!ls -lh logs/gtq/weights/

## 第 8 步：提取音色索引（提升相似度）

In [ ]:
%cd /kaggle/working/rvc
!python train/train_index.py gtq v2 logs 2 2>&1 | tail -5
print("索引提取完成")
!ls -lh logs/gtq/

## 第 9 步：本地试听（可选）

先在本机用微软 TTS 生成一段男声情绪底音，再上传到这里转换。
- 本机生成底音：`python edge_voice.py 我的台词.txt`（桌面工具）
- 在本 notebook **左侧文件栏（File）** 上传底音 mp3（会出现在 /kaggle/working）

In [ ]:
%cd /kaggle/working/rvc
import glob, os, shutil, subprocess

# 找到上传的底音（搜 /kaggle/working 下的 mp3/wav/m4a）
cands = glob.glob("/kaggle/working/*.mp3") + glob.glob("/kaggle/working/*.wav") + glob.glob("/kaggle/working/*.m4a")
print("找到底音:", cands)
if not cands:
    print("请先在左侧 文件(File) 面板上传一个音频到 /kaggle/working 再重跑本格")
    raise SystemExit

weights = sorted(glob.glob("logs/gtq/weights/gtq_e*.pth"))
print("模型文件:", [os.path.basename(w) for w in weights])
shutil.copy(weights[-1], "assets/weights/gtq.pth")

test_in = cands[0]
!python infer/cli.py --model gtq --input "{test_in}" --output test_output \
    --f0-method rmvpe --index-rate 0.75 --format wav 2>&1 | tail -3

out = glob.glob("test_output/*.wav")
print("转换完成:", out)

## 第 10 步：下载训练好的模型（之后无限次使用）

在 **左侧文件栏（File）** 找到 `/kaggle/working/rvc/logs/gtq/weights/gtq_e*.pth`，
以及 `/kaggle/working/rvc/logs/gtq/` 下的 `added_*.index` 或 `trained_*.index`，
**勾选后点下载**保存到桌面「中转站」文件夹。

> 若文件名是 gtq_e300_s250.pth 这种，下载那个 epoch 最大的即可（类似 250 这种大的数字）。

In [ ]:
%cd /kaggle/working/rvc
import glob, os
print("模型文件（下载 epoch 最大的）：")
for w in sorted(glob.glob("logs/gtq/weights/gtq_e*.pth")):
    print("  ", os.path.basename(w), f"{os.path.getsize(w)/1e6:.1f} MB")
idx = glob.glob("logs/gtq/added_*.index") or glob.glob("logs/gtq/trained_*.index")
print("索引文件（也需下载）：")
for i in idx:
    print("  ", os.path.basename(i))